In [1]:
import pandas as pd


In [2]:
# INPUTS
SEASON = '2024-2025'
LEAGUE_SLUG = 'Premier-League'

In [3]:
team_style_path = f'../../data/features/context/team_style_match_{SEASON}_{LEAGUE_SLUG}.parquet'

ts = pd.read_parquet(team_style_path)


In [4]:
fixtures = pd.read_csv(f'../../data/raw/fbref/fixtures_{SEASON}_{LEAGUE_SLUG}.csv')
team_stats = pd.read_csv(f'../../data/raw/fbref/team_stats_{SEASON}_{LEAGUE_SLUG}.csv')

In [5]:
team_stats[['match_id','home_shots','away_shots']]

,match_id,home_shots,away_shots
0,cc5b4244,14,10
1,a1d0d529,7,18
2,34557647,3,19
3,71618ace,9,10
4,4efc72e4,14,13
...,...,...,...
375,464cbad6,10,6
376,3d22336e,13,19
377,36844e73,17,14
378,1ff370e8,20,3


In [8]:
ts


# ['exp_pace_home'].describe()

,match_date,home_team,away_team,match_id,home_shots_per_poss_for_w8,home_shots_per_poss_against_w8,home_sot_rate_for_w8,home_sot_rate_against_w8,home_poss_for_w8,home_poss_std_w8,away_shots_per_poss_for_w8,away_shots_per_poss_against_w8,away_sot_rate_for_w8,away_sot_rate_against_w8,away_poss_for_w8,away_poss_std_w8
0,2024-08-16,Manchester Utd,Fulham,cc5b4244,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-08-17,Ipswich Town,Liverpool,a1d0d529,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-08-17,Newcastle Utd,Southampton,34557647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-08-17,Everton,Brighton,71618ace,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-08-17,Nott'ham Forest,Bournemouth,4efc72e4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,2025-05-25,Nott'ham Forest,Chelsea,464cbad6,25.308706,29.067531,0.389380,0.323795,44.250,9.437766,28.824638,21.492342,0.385129,0.284800,54.250,10.593124
376,2025-05-25,Fulham,Manchester City,3d22336e,26.183455,24.531489,0.352400,0.402158,51.500,10.281745,23.021406,17.991053,0.366949,0.170272,64.625,5.829420
377,2025-05-25,Newcastle Utd,Everton,36844e73,27.878741,21.487152,0.348689,0.308238,53.375,10.966670,20.269147,21.452754,0.337959,0.405690,42.125,12.052712
378,2025-05-25,Bournemouth,Leicester City,1ff370e8,23.265828,23.231539,0.265941,0.322628,52.000,12.660399,19.050586,32.780972,0.255952,0.250727,44.625,8.650970


In [ ]:
df = fixtures.merge(team_stats[["match_id","home_shots","away_shots"]], on="match_id", how="inner")
df = df.merge(ts[["match_id",'home_shots_per_poss_for_w8', 'home_shots_per_poss_against_w8',
       'home_sot_rate_for_w8', 'home_sot_rate_against_w8', 'home_poss_for_w8',
       'home_poss_std_w8', 'away_shots_per_poss_for_w8',
       'away_shots_per_poss_against_w8', 'away_sot_rate_for_w8',
       'away_sot_rate_against_w8', 'away_poss_for_w8', 'away_poss_std_w8']], on="match_id", how="inner")
# df["shots_total"] = df["home_shots"] + df["away_shots"]

df[["exp_pace_total","shots_total"]].corr()           # correlation
# (df["shots_total"] - df["exp_pace_total"]).describe() # residuals


,exp_pace_total,shots_total
exp_pace_total,1.000000,0.193671
shots_total,0.193671,1.000000


In [9]:
bins = pd.qcut(df["exp_pace_total"], 10, duplicates="drop")
df.groupby(bins)["shots_total"].mean().to_frame("obs").assign(pred=df.groupby(bins)["exp_pace_total"].mean())


/var/folders/l_/pcb32qsn4vz5n55rmw_4dctc0000gn/T/ipykernel_42351/2236549888.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(bins)["shots_total"].mean().to_frame("obs").assign(pred=df.groupby(bins)["exp_pace_total"].mean())


,obs,pred
exp_pace_total,,
"(19.775, 22.768]",23.297297,22.018740
"(22.768, 23.775]",24.405405,23.277991
"(23.775, 24.436]",25.432432,24.165481
"(24.436, 25.0]",25.756757,24.752649
"(25.0, 25.655]",24.864865,25.347059
"(25.655, 26.203]",25.702703,25.900920
"(26.203, 26.93]",26.648649,26.544730
"(26.93, 27.62]",26.756757,27.283577
"(27.62, 28.357]",26.837838,27.947274
